# Stage 6 — One-Time Target Unsealing and Frozen External Validation

**Authorised scope:** verify every Stage 5B pre-unseal commitment, seal this executable Stage 6 analysis, then open exactly the two committed sealed-label files and evaluate only the already frozen DeepDRiD target scores.

**User authorisation:** the user explicitly authorised Stage 6 after the Stage 5B prediction-freeze completion was reported, while requiring the model, threshold, and Stage 5B predictions to remain frozen and prohibiting post-unseal tuning.

This notebook does not train, refit, recalibrate, reverse, select, or adapt any model. It does not optimise a threshold. The raw target orientation and fixed probability threshold of 0.5 are retained. AUC is the primary endpoint; all other analyses are secondary or exploratory and cannot change the primary result.


In [1]:
#@title 06-0. Verify Stage 5B freeze and seal Stage 6 before opening labels
from google.colab import drive
drive.mount("/content/drive")

import hashlib
import importlib.metadata
import json
import os
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


print("================ STAGE 6 PRE-UNSEAL VERIFICATION ================")

PROJECT_ROOT = Path("/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability")
CODE_ROOT = PROJECT_ROOT / "05_Code" / "Retinal_DR"
TEST_ROOT = (
    PROJECT_ROOT / "06_Data_Records" / "Retinal_DR" /
    "Prospective_Retinal_Blind_Test_v0.1"
)
STAGE2_ROOT = TEST_ROOT / "Stage2_Acquisition_And_Quarantine_v0.1"
STAGE5B_ROOT = (
    TEST_ROOT /
    "Stage5B_Label_Free_Target_Scoring_And_Prediction_Freeze_v0.1"
)
STAGE6_ROOT = (
    TEST_ROOT /
    "Stage6_One_Time_Target_Unsealing_And_External_Validation_v0.1"
)
PROTOCOL_ROOT = STAGE6_ROOT / "00_Protocol"
EVALUATION_ROOT = STAGE6_ROOT / "01_Unsealed_Evaluation"
RESULT_ROOT = STAGE6_ROOT / "02_Results"
for directory in [PROTOCOL_ROOT, EVALUATION_ROOT, RESULT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = (
    CODE_ROOT /
    "Retinal_DR_Stage6_One_Time_Unsealing_And_External_Validation_v0.1_R1.ipynb"
)
STAGE5B_FINAL_PATH = (
    STAGE5B_ROOT / "02_Prediction_Freeze" /
    "Stage5B_Prediction_Freeze_Complete_v0.1.json"
)
STAGE5B_INTEGRITY_PATH = (
    STAGE5B_ROOT / "02_Prediction_Freeze" /
    "Stage5B_PreUnseal_Output_Integrity_Manifest_v0.1.csv"
)
STAGE2_LABEL_COMMITMENT_PATH = (
    STAGE2_ROOT / "Stage2_Sealed_Label_Commitments_v0.1.json"
)
QUARANTINE_ROOT = (
    TEST_ROOT / "99_Label_Quarantine_DO_NOT_OPEN_BEFORE_STAGE6"
)
SEALED_LABEL_ROOT = QUARANTINE_ROOT / "01_Sealed_Evaluation_Labels"
LABEL_PATHS = {
    "APTOS_2019": (
        SEALED_LABEL_ROOT /
        "APTOS_2019_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
    ),
    "IDRiD": (
        SEALED_LABEL_ROOT /
        "IDRiD_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
    ),
}
TARGET_SCORE_PATHS = {
    target: (
        STAGE5B_ROOT / "01_Label_Free_Scores" /
        f"{target}_Frozen_DeepDRiD_Axis_Image_Scores_v0.1.csv"
    )
    for target in LABEL_PATHS
}
EDGE_RECORD_PATHS = {
    "APTOS_2019": (
        STAGE5B_ROOT / "02_Prediction_Freeze" /
        "DDR_APTOS_v0.1_PreUnseal_Prediction_Record_v0.1.json"
    ),
    "IDRiD": (
        STAGE5B_ROOT / "02_Prediction_Freeze" /
        "DDR_IDRiD_v0.1_PreUnseal_Prediction_Record_v0.1.json"
    ),
}

PROTOCOL_SEAL_PATH = PROTOCOL_ROOT / "Stage6_Executable_Unsealing_Protocol_Seal_v0.1.json"
RUNTIME_STATE_PATH = RESULT_ROOT / "Stage6_Runtime_Unsealing_State_v0.1.json"
FINAL_DECISION_PATH = RESULT_ROOT / "Stage6_External_Validation_Complete_v0.1.json"

USER_AUTHORISATION = (
    "User explicitly authorised entry into Stage 6 after receiving the "
    "Stage 5B freeze result; model, threshold, and Stage 5B predictions "
    "must remain frozen, with no post-unseal tuning."
)
EXPECTED_STAGE5B_FREEZE_HASH = (
    "f263225b6e44c7fc3b7709533afd5aae8b04d47b7ec77dc8d0c5bac911e5f1b9"
)
EXPECTED_STAGE5B_INTEGRITY_HASH = (
    "33531c1d2d7bc26222d451b0720d96e3f0fd4473e25f25f9d7a4475d06cdd50b"
)
EXPECTED_LABEL_HASHES = {
    "APTOS_2019": "cc4fdbdd0d17e0696bcd569e082a99b266dd177d6c721344dbb7a51d19687c13",
    "IDRiD": "361c2dfc7b362d60b6ce374922c2373889a8286ce904ada8515641262c8734ae",
}
EXPECTED_LABEL_ROWS = {"APTOS_2019": 1080, "IDRiD": 102}
RANDOM_SEED = 20260721
N_BOOTSTRAP = 2000
FIXED_PROBABILITY_THRESHOLD = 0.50
AUC_FEASIBILITY_MINIMUM = 0.70
AUC_CI_LOWER_STRICT_MINIMUM = 0.55
ENDPOINT_ID = "MODERATE_OR_WORSE_DR_GRADE_GE_2"
POSITIVE_CLASS_RULE = "original DR grade >= 2"


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sha256_json(payload):
    encoded = json.dumps(
        payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def atomic_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
    os.replace(temporary, path)


def normalised_notebook_source_sha256(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        notebook = json.load(handle)
    cells = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") not in {"code", "markdown"}:
            continue
        cell_source = cell.get("source", [])
        if isinstance(cell_source, list):
            cell_source = "".join(cell_source)
        cells.append({
            "cell_type": cell.get("cell_type"),
            "source": str(cell_source).replace("\r\n", "\n"),
        })
    return sha256_json(cells)


required_pre_unseal_paths = [
    NOTEBOOK_PATH,
    STAGE5B_FINAL_PATH,
    STAGE5B_INTEGRITY_PATH,
    STAGE2_LABEL_COMMITMENT_PATH,
    *TARGET_SCORE_PATHS.values(),
    *EDGE_RECORD_PATHS.values(),
]
for path in required_pre_unseal_paths:
    assert path.is_file(), f"Missing required frozen input: {path}"

with STAGE5B_FINAL_PATH.open("r", encoding="utf-8") as handle:
    stage5b_final = json.load(handle)
claimed_freeze_hash = stage5b_final["freeze_record_sha256"]
stage5b_without_claim = dict(stage5b_final)
stage5b_without_claim.pop("freeze_record_sha256")
assert sha256_json(stage5b_without_claim) == claimed_freeze_hash
assert claimed_freeze_hash == EXPECTED_STAGE5B_FREEZE_HASH
assert stage5b_final["decision"] == (
    "PREDICTIONS_FROZEN_ADVANCE_TO_SEPARATELY_GOVERNED_STAGE6_UNSEALING"
)
assert stage5b_final["target_sealed_label_files_accessed"] is False
assert stage5b_final["target_performance_observed"] is False
assert stage5b_final["unsealing_authorised_inside_this_notebook"] is False
assert sha256_file(STAGE5B_INTEGRITY_PATH) == EXPECTED_STAGE5B_INTEGRITY_HASH
assert stage5b_final["output_integrity_manifest_sha256"] == EXPECTED_STAGE5B_INTEGRITY_HASH

integrity_manifest = pd.read_csv(STAGE5B_INTEGRITY_PATH)
assert list(integrity_manifest.columns) == ["relative_path", "size_bytes", "sha256"]
EXPECTED_STAGE5B_INTEGRITY_ITEMS = 20
assert len(integrity_manifest) == EXPECTED_STAGE5B_INTEGRITY_ITEMS
stage5b_root_resolved = STAGE5B_ROOT.resolve()
for row in integrity_manifest.itertuples(index=False):
    candidate = (STAGE5B_ROOT / str(row.relative_path)).resolve()
    assert stage5b_root_resolved in candidate.parents
    assert candidate.is_file(), candidate
    assert int(candidate.stat().st_size) == int(row.size_bytes)
    assert sha256_file(candidate) == str(row.sha256)

edge_records = {}
for target, path in EDGE_RECORD_PATHS.items():
    with path.open("r", encoding="utf-8") as handle:
        record = json.load(handle)
    expected_edge_hash = stage5b_final["edge_prediction_record_sha256"][record["edge_id"]]
    assert sha256_file(path) == expected_edge_hash
    assert record["target"] == target
    assert record["source"] == "DeepDRiD"
    assert record["target_labels_accessed"] is False
    assert record["target_performance_observed"] is False
    assert record["abstain"] is True
    assert record["transfer_risk"] == "NOT_CALIBRATED_OUT_OF_SUPPORT"
    edge_records[target] = record

with STAGE2_LABEL_COMMITMENT_PATH.open("r", encoding="utf-8") as handle:
    label_commitments = json.load(handle)
assert label_commitments["endpoint_id"] == ENDPOINT_ID
commitment_keys = {
    "APTOS_2019": "aptos_sealed_evaluation",
    "IDRiD": "idrid_sealed_evaluation",
}
for target, key in commitment_keys.items():
    commitment = label_commitments[key]
    assert int(commitment["rows"]) == EXPECTED_LABEL_ROWS[target]
    assert str(commitment["sha256"]) == EXPECTED_LABEL_HASHES[target]
    assert int(commitment["authorised_open_stage"]) == 6
    assert commitment["displayed"] is False
    assert Path(commitment["path"]).resolve() == LABEL_PATHS[target].resolve()

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "numpy": importlib.metadata.version("numpy"),
    "pandas": importlib.metadata.version("pandas"),
    "scikit_learn": importlib.metadata.version("scikit-learn"),
    "scipy": importlib.metadata.version("scipy"),
    "matplotlib": importlib.metadata.version("matplotlib"),
}
analysis_spec = {
    "primary_endpoint": "raw-orientation image-level ROC AUC",
    "primary_uncertainty": (
        "2000 deterministic stratified image bootstraps, positive and negative "
        "records resampled separately at observed class counts"
    ),
    "feasibility_floor": {
        "auc_at_or_above": AUC_FEASIBILITY_MINIMUM,
        "auc_ci_95_lower_strictly_above": AUC_CI_LOWER_STRICT_MINIMUM,
    },
    "secondary_metrics": [
        "average_precision", "Brier_score", "log_loss", "ECE_10_equal_frequency",
        "accuracy", "balanced_accuracy", "sensitivity", "specificity",
        "PPV", "NPV", "F1_at_fixed_probability_0.5",
    ],
    "prevalence_interval": "Clopper-Pearson exact 95% interval",
    "orientation": "retain frozen raw axis; never replace AUC by max(AUC, 1-AUC)",
    "threshold": FIXED_PROBABILITY_THRESHOLD,
    "threshold_tuning": "prohibited",
    "model_refitting_or_adaptation": "prohibited",
    "target_calibration_fitting": "prohibited",
    "relative_edge_ordering": "not evaluated because Stage 5B precommitted none for n=2",
    "failure_mode_probe": (
        "exploratory only: on the frozen 64-image target anchors, compare absolute "
        "dominant-transform logit change between fixed-threshold errors and correct "
        "predictions; it cannot alter the primary conclusion"
    ),
}
input_hashes = {
    "notebook_source_sha256": normalised_notebook_source_sha256(NOTEBOOK_PATH),
    "stage5b_final_file_sha256": sha256_file(STAGE5B_FINAL_PATH),
    "stage5b_internal_freeze_sha256": claimed_freeze_hash,
    "stage5b_integrity_manifest_sha256": sha256_file(STAGE5B_INTEGRITY_PATH),
    "stage2_label_commitment_sha256": sha256_file(STAGE2_LABEL_COMMITMENT_PATH),
    "analysis_spec_sha256": sha256_json(analysis_spec),
    "environment_sha256": sha256_json(environment),
    "target_score_sha256": {
        target: sha256_file(path) for target, path in TARGET_SCORE_PATHS.items()
    },
    "edge_record_sha256": {
        target: sha256_file(path) for target, path in EDGE_RECORD_PATHS.items()
    },
    "committed_label_sha256": EXPECTED_LABEL_HASHES,
}
seal_payload = {
    "stage": "Stage6",
    "decision": "AUTHORISED_ONE_TIME_UNSEALING_AFTER_STAGE5B_PREDICTION_FREEZE",
    "user_authorisation": USER_AUTHORISATION,
    "authorised_label_files": {target: str(path) for target, path in LABEL_PATHS.items()},
    "authorised_label_rows": EXPECTED_LABEL_ROWS,
    "analysis_spec": analysis_spec,
    "input_hashes": input_hashes,
    "labels_opened_before_seal": False,
    "sealed_utc": utc_now(),
}
seal_payload["seal_sha256"] = sha256_json(seal_payload)
if PROTOCOL_SEAL_PATH.is_file():
    with PROTOCOL_SEAL_PATH.open("r", encoding="utf-8") as handle:
        existing_seal = json.load(handle)
    existing_without_claim = dict(existing_seal)
    existing_claim = existing_without_claim.pop("seal_sha256")
    assert sha256_json(existing_without_claim) == existing_claim
    assert existing_seal["input_hashes"] == input_hashes
    assert existing_seal["analysis_spec"] == analysis_spec
    seal_payload = existing_seal
else:
    atomic_json(PROTOCOL_SEAL_PATH, seal_payload)

runtime_state = {
    "stage6_protocol_seal_sha256": seal_payload["seal_sha256"],
    "stage5b_freeze_sha256": claimed_freeze_hash,
    "target_sealed_label_files_accessed": False,
    "target_performance_observed": False,
    "model_refit": False,
    "threshold_tuned": False,
    "last_updated_utc": utc_now(),
}
atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("Stage 5B freeze verified:", claimed_freeze_hash)
print("Stage 5B integrity items verified:", len(integrity_manifest))
print("Stage 6 protocol seal:", seal_payload["seal_sha256"])
print("Target sealed-label files accessed before seal: False")
print("Target performance observed before seal: False")
print("Model/refit/threshold changes authorised: False")


Mounted at /content/drive
================ STAGE 6 PRE-UNSEAL VERIFICATION ================
Stage 5B freeze verified: f263225b6e44c7fc3b7709533afd5aae8b04d47b7ec77dc8d0c5bac911e5f1b9
Stage 5B integrity items verified: 20
Stage 6 protocol seal: 8b3b5b1bd85ed25939bf17f2f54502829af2c9437f5e71fd86bd0842b258cec6
Target sealed-label files accessed before seal: False
Target performance observed before seal: False
Model/refit/threshold changes authorised: False


In [2]:
#@title 06-1. One-time authorised unsealing, exact join, and frozen evaluation
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    roc_auc_score,
    roc_curve,
)
from scipy.stats import beta


print("================ STAGE 6 AUTHORISED LABEL OPEN ================")
assert PROTOCOL_SEAL_PATH.is_file()
assert seal_payload["labels_opened_before_seal"] is False
assert seal_payload["seal_sha256"] == sha256_json({
    key: value for key, value in seal_payload.items() if key != "seal_sha256"
})

authorised_label_paths = {str(path.resolve()) for path in LABEL_PATHS.values()}
quarantine_root_resolved = str(QUARANTINE_ROOT.resolve())
label_open_events = []


def _stage6_quarantine_guard(event, args):
    if event != "open" or not args:
        return
    candidate = args[0]
    if not isinstance(candidate, (str, bytes, os.PathLike)):
        return
    resolved = str(Path(candidate).resolve())
    if resolved.startswith(quarantine_root_resolved + os.sep):
        if resolved not in authorised_label_paths:
            raise PermissionError(
                "Stage 6 permits only the two committed sealed-evaluation label files: "
                + resolved
            )
        label_open_events.append(resolved)


sys.addaudithook(_stage6_quarantine_guard)
runtime_state.update({
    "target_sealed_label_files_accessed": True,
    "target_performance_observed": False,
    "label_access_started_utc": utc_now(),
    "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)


def calibration_table(labels, probabilities, bins=10):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    order = np.argsort(probabilities, kind="mergesort")
    bin_ids = np.empty(len(labels), dtype=np.int64)
    bin_ids[order] = np.minimum(
        np.floor(np.arange(len(labels)) * bins / len(labels)).astype(int), bins - 1
    )
    rows = []
    for bin_id in range(bins):
        mask = bin_ids == bin_id
        if not np.any(mask):
            continue
        rows.append({
            "bin": int(bin_id + 1),
            "n": int(mask.sum()),
            "mean_probability": float(probabilities[mask].mean()),
            "observed_fraction_positive": float(labels[mask].mean()),
        })
    return pd.DataFrame(rows)


def expected_calibration_error(labels, probabilities, bins=10):
    table = calibration_table(labels, probabilities, bins=bins)
    return float(
        np.sum(
            table["n"] / table["n"].sum() *
            np.abs(table["mean_probability"] - table["observed_fraction_positive"])
        )
    )


def point_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.clip(np.asarray(probabilities, dtype=np.float64), 1e-12, 1 - 1e-12)
    predictions = (probabilities >= FIXED_PROBABILITY_THRESHOLD).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    ppv = tp / (tp + fp) if (tp + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    return {
        "roc_auc": float(roc_auc_score(labels, probabilities)),
        "average_precision": float(average_precision_score(labels, probabilities)),
        "brier_score": float(brier_score_loss(labels, probabilities)),
        "log_loss": float(log_loss(labels, probabilities, labels=[0, 1])),
        "ece_10": expected_calibration_error(labels, probabilities, bins=10),
        "accuracy_at_0_5": float(accuracy_score(labels, predictions)),
        "balanced_accuracy_at_0_5": float((sensitivity + specificity) / 2),
        "sensitivity_at_0_5": float(sensitivity),
        "specificity_at_0_5": float(specificity),
        "ppv_at_0_5": float(ppv),
        "npv_at_0_5": float(npv),
        "f1_at_0_5": float(f1_score(labels, predictions, zero_division=0)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


BOOTSTRAP_KEYS = [
    "roc_auc", "average_precision", "brier_score", "log_loss", "ece_10",
    "accuracy_at_0_5", "balanced_accuracy_at_0_5", "sensitivity_at_0_5",
    "specificity_at_0_5", "ppv_at_0_5", "npv_at_0_5", "f1_at_0_5",
]


def stratified_bootstrap(labels, probabilities, seed):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    positive = np.flatnonzero(labels == 1)
    negative = np.flatnonzero(labels == 0)
    assert len(positive) >= 2 and len(negative) >= 2
    rng = np.random.default_rng(seed)
    distributions = {key: np.empty(N_BOOTSTRAP, dtype=np.float64) for key in BOOTSTRAP_KEYS}
    for bootstrap_index in range(N_BOOTSTRAP):
        sampled = np.concatenate([
            rng.choice(negative, size=len(negative), replace=True),
            rng.choice(positive, size=len(positive), replace=True),
        ])
        values = point_metrics(labels[sampled], probabilities[sampled])
        for key in BOOTSTRAP_KEYS:
            distributions[key][bootstrap_index] = values[key]
    return {
        key: [
            float(np.nanquantile(values, 0.025)),
            float(np.nanquantile(values, 0.975)),
        ]
        for key, values in distributions.items()
    }


def clopper_pearson(successes, total, alpha=0.05):
    lower = 0.0 if successes == 0 else float(beta.ppf(alpha / 2, successes, total - successes + 1))
    upper = 1.0 if successes == total else float(beta.ppf(1 - alpha / 2, successes + 1, total - successes))
    return [lower, upper]


def finite_or_none(value):
    value = float(value)
    return value if np.isfinite(value) else None


evaluation_tables = {}
metric_records = {}
calibration_tables = {}
for target_index, target in enumerate(["APTOS_2019", "IDRiD"]):
    label_path = LABEL_PATHS[target]

    # First byte-level access after the executable Stage 6 protocol seal.
    observed_label_hash = sha256_file(label_path)
    assert observed_label_hash == EXPECTED_LABEL_HASHES[target]
    labels = pd.read_csv(label_path, dtype={"image_id": str})
    assert list(labels.columns) == [
        "dataset", "image_id", "original_dr_grade", "moderate_or_worse_dr"
    ]
    assert len(labels) == EXPECTED_LABEL_ROWS[target]
    assert labels["image_id"].is_unique
    assert labels["dataset"].astype(str).eq(target).all()
    assert set(labels["moderate_or_worse_dr"].astype(int).unique()) == {0, 1}
    assert labels["original_dr_grade"].between(0, 4).all()
    assert np.array_equal(
        labels["moderate_or_worse_dr"].astype(int).to_numpy(),
        labels["original_dr_grade"].ge(2).astype(int).to_numpy(),
    )

    scores = pd.read_csv(TARGET_SCORE_PATHS[target], dtype={"image_id": str})
    assert list(scores.columns) == [
        "dataset", "image_id", "canonical_sha256",
        "frozen_axis_logit", "frozen_axis_probability",
    ]
    assert len(scores) == EXPECTED_LABEL_ROWS[target]
    assert scores["image_id"].is_unique
    assert set(scores["image_id"]) == set(labels["image_id"])
    assert np.isfinite(scores[["frozen_axis_logit", "frozen_axis_probability"]]).all().all()
    assert scores["frozen_axis_probability"].between(0, 1).all()

    joined = scores.merge(
        labels,
        on=["dataset", "image_id"],
        how="inner",
        validate="one_to_one",
        sort=True,
    )
    assert len(joined) == EXPECTED_LABEL_ROWS[target]
    joined["fixed_prediction_at_0_5"] = (
        joined["frozen_axis_probability"] >= FIXED_PROBABILITY_THRESHOLD
    ).astype(int)
    joined["fixed_prediction_correct"] = (
        joined["fixed_prediction_at_0_5"] == joined["moderate_or_worse_dr"].astype(int)
    )

    y = joined["moderate_or_worse_dr"].astype(int).to_numpy()
    p = joined["frozen_axis_probability"].astype(float).to_numpy()
    metrics = point_metrics(y, p)
    intervals = stratified_bootstrap(y, p, RANDOM_SEED + 100 * target_index)
    prevalence = float(y.mean())
    prevalence_ci = clopper_pearson(int(y.sum()), len(y))
    primary_pass = bool(
        metrics["roc_auc"] >= AUC_FEASIBILITY_MINIMUM and
        intervals["roc_auc"][0] > AUC_CI_LOWER_STRICT_MINIMUM
    )
    warning_evaluation = (
        "OBSERVED_DISCRIMINATION_MEETS_FLOOR_DESPITE_CONSERVATIVE_ABSTENTION"
        if primary_pass else
        "OBSERVED_DISCRIMINATION_BELOW_PRECOMMITTED_FEASIBILITY_FLOOR"
    )

    record = {
        "edge_id": edge_records[target]["edge_id"],
        "source": "DeepDRiD",
        "target": target,
        "endpoint_id": ENDPOINT_ID,
        "positive_class_rule": POSITIVE_CLASS_RULE,
        "evaluation_unit": "image; no authenticated patient/eye grouping field exists in sealed schema",
        "n": int(len(y)),
        "negative": int((y == 0).sum()),
        "positive": int((y == 1).sum()),
        "prevalence": prevalence,
        "prevalence_ci_95_exact": prevalence_ci,
        "fixed_probability_threshold": FIXED_PROBABILITY_THRESHOLD,
        "metrics": {
            key: (int(value) if key in {"tn", "fp", "fn", "tp"} else finite_or_none(value))
            for key, value in metrics.items()
        },
        "metric_ci_95": {
            key: [finite_or_none(bounds[0]), finite_or_none(bounds[1])]
            for key, bounds in intervals.items()
        },
        "source_validation_auc_reference": float(
            edge_records[target]["baselines"]["source_validation_eye_auc"]
        ),
        "source_minus_target_auc_descriptive": float(
            edge_records[target]["baselines"]["source_validation_eye_auc"] - metrics["roc_auc"]
        ),
        "stage5b_transfer_risk": edge_records[target]["transfer_risk"],
        "stage5b_abstain": bool(edge_records[target]["abstain"]),
        "stage5b_dominant_failure_mode": edge_records[target]["dominant_failure_mode"],
        "meets_precommitted_external_feasibility_floor": primary_pass,
        "stage5b_warning_evaluation": warning_evaluation,
        "model_refit": False,
        "threshold_tuned": False,
        "orientation_reversed": False,
        "label_sha256": observed_label_hash,
        "score_sha256": sha256_file(TARGET_SCORE_PATHS[target]),
        "stage6_protocol_seal_sha256": seal_payload["seal_sha256"],
    }
    evaluation_tables[target] = joined
    metric_records[target] = record
    calibration_tables[target] = calibration_table(y, p, bins=10).assign(target=target)

assert set(label_open_events) == authorised_label_paths
runtime_state.update({
    "target_sealed_label_files_accessed": True,
    "target_performance_observed": True,
    "authorised_label_files_opened": sorted(set(label_open_events)),
    "model_refit": False,
    "threshold_tuned": False,
    "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)

summary_rows = []
for target, record in metric_records.items():
    metrics = record["metrics"]
    intervals = record["metric_ci_95"]
    summary_rows.append({
        "edge_id": record["edge_id"],
        "target": target,
        "n": record["n"],
        "positive": record["positive"],
        "prevalence": record["prevalence"],
        "roc_auc": metrics["roc_auc"],
        "roc_auc_ci_95_lower": intervals["roc_auc"][0],
        "roc_auc_ci_95_upper": intervals["roc_auc"][1],
        "average_precision": metrics["average_precision"],
        "balanced_accuracy_at_0_5": metrics["balanced_accuracy_at_0_5"],
        "sensitivity_at_0_5": metrics["sensitivity_at_0_5"],
        "specificity_at_0_5": metrics["specificity_at_0_5"],
        "brier_score": metrics["brier_score"],
        "ece_10": metrics["ece_10"],
        "stage5b_abstain": record["stage5b_abstain"],
        "meets_external_feasibility_floor": record[
            "meets_precommitted_external_feasibility_floor"
        ],
        "warning_evaluation": record["stage5b_warning_evaluation"],
    })
stage6_summary = pd.DataFrame(summary_rows)
display(stage6_summary)
print("Target sealed-label files accessed: True (authorised Stage 6 only)")
print("Model refit: False")
print("Threshold tuned: False")


================ STAGE 6 AUTHORISED LABEL OPEN ================


,edge_id,target,n,positive,prevalence,roc_auc,roc_auc_ci_95_lower,roc_auc_ci_95_upper,average_precision,balanced_accuracy_at_0_5,sensitivity_at_0_5,specificity_at_0_5,brier_score,ece_10,stage5b_abstain,meets_external_feasibility_floor,warning_evaluation
0,DDR_APTOS_v0.1,APTOS_2019,1080,432,0.400000,0.678434,0.644969,0.711861,0.640803,0.625772,0.634259,0.617284,0.323734,0.300644,True,False,OBSERVED_DISCRIMINATION_BELOW_PRECOMMITTED_FEA...
1,DDR_IDRiD_v0.1,IDRiD,102,63,0.617647,0.782662,0.692715,0.862434,0.865760,0.636752,0.888889,0.384615,0.261180,0.251944,True,True,OBSERVED_DISCRIMINATION_MEETS_FLOOR_DESPITE_CO...


Target sealed-label files accessed: True (authorised Stage 6 only)
Model refit: False
Threshold tuned: False


In [3]:
#@title 06-2. Frozen failure-mode probe, figures, records, and final Stage 6 decision
import matplotlib.pyplot as plt


def deterministic_csv(path, frame):
    text = frame.to_csv(index=False, lineterminator="\n", float_format="%.12g")
    if path.is_file():
        assert path.read_text(encoding="utf-8") == text, f"Existing output changed: {path}"
    else:
        path.write_text(text, encoding="utf-8")


def deterministic_json(path, payload):
    if path.is_file():
        with path.open("r", encoding="utf-8") as handle:
            existing = json.load(handle)
        assert existing == payload, f"Existing output changed: {path}"
    else:
        atomic_json(path, payload)


def deterministic_text(path, text):
    if path.is_file():
        assert path.read_text(encoding="utf-8") == text, f"Existing output changed: {path}"
    else:
        path.write_text(text, encoding="utf-8")


transform_path = (
    STAGE5B_ROOT / "01_Label_Free_Scores" /
    "Stage5B_Frozen_Transformation_Image_Logits_v0.1.csv"
)
transform_table = pd.read_csv(transform_path, dtype={"image_id": str})
mechanism_rows = []
for target_index, target in enumerate(["APTOS_2019", "IDRiD"]):
    dominant_transform = edge_records[target]["dominant_transform"]
    transformed = transform_table[
        transform_table["dataset"].eq(target) &
        transform_table["transform_name"].eq(dominant_transform)
    ][["image_id", "transformed_logit"]].copy()
    joined = evaluation_tables[target][[
        "image_id", "frozen_axis_logit", "frozen_axis_probability",
        "moderate_or_worse_dr", "fixed_prediction_correct",
    ]].merge(transformed, on="image_id", how="inner", validate="one_to_one")
    assert len(joined) == 64
    joined["absolute_dominant_transform_logit_change"] = np.abs(
        joined["transformed_logit"] - joined["frozen_axis_logit"]
    )
    sensitivity = joined["absolute_dominant_transform_logit_change"].to_numpy(float)
    correct = joined["fixed_prediction_correct"].to_numpy(bool)
    error_mask = ~correct
    gap = (
        float(np.median(sensitivity[error_mask]) - np.median(sensitivity[correct]))
        if np.any(error_mask) and np.any(correct) else np.nan
    )
    high_threshold = float(np.quantile(sensitivity, 0.75))
    high = sensitivity >= high_threshold
    high_error_rate = float(error_mask[high].mean()) if np.any(high) else np.nan
    other_error_rate = float(error_mask[~high].mean()) if np.any(~high) else np.nan
    rng = np.random.default_rng(RANDOM_SEED + 500 + target_index)
    bootstrap_gap = []
    for _ in range(N_BOOTSTRAP):
        sampled = rng.integers(0, len(joined), size=len(joined))
        sampled_error = error_mask[sampled]
        sampled_sensitivity = sensitivity[sampled]
        if np.any(sampled_error) and np.any(~sampled_error):
            bootstrap_gap.append(
                np.median(sampled_sensitivity[sampled_error]) -
                np.median(sampled_sensitivity[~sampled_error])
            )
    gap_ci = (
        [float(np.quantile(bootstrap_gap, 0.025)), float(np.quantile(bootstrap_gap, 0.975))]
        if bootstrap_gap else [None, None]
    )
    mechanism_rows.append({
        "edge_id": edge_records[target]["edge_id"],
        "target": target,
        "stage5b_dominant_transform": dominant_transform,
        "stage5b_dominant_failure_mode": edge_records[target]["dominant_failure_mode"],
        "anchor_images": int(len(joined)),
        "fixed_threshold_errors": int(error_mask.sum()),
        "median_abs_logit_change_errors_minus_correct": gap,
        "gap_ci_95_exploratory": json.dumps(gap_ci),
        "top_quartile_sensitivity_error_rate": high_error_rate,
        "remaining_sensitivity_error_rate": other_error_rate,
        "interpretation_scope": "EXPLORATORY_MECHANISM_PROBE_NOT_PRIMARY_VALIDATION",
    })
mechanism_table = pd.DataFrame(mechanism_rows)

for target, table in evaluation_tables.items():
    deterministic_csv(
        EVALUATION_ROOT / f"{target}_Frozen_Score_And_Unsealed_Label_Evaluation_v0.1.csv",
        table,
    )
    deterministic_json(
        EVALUATION_ROOT / f"{edge_records[target]['edge_id']}_External_Validation_Record_v0.1.json",
        metric_records[target],
    )
deterministic_csv(
    EVALUATION_ROOT / "Stage6_External_Validation_Summary_v0.1.csv",
    stage6_summary,
)
deterministic_csv(
    EVALUATION_ROOT / "Stage6_Calibration_Bins_v0.1.csv",
    pd.concat(calibration_tables.values(), ignore_index=True)[
        ["target", "bin", "n", "mean_probability", "observed_fraction_positive"]
    ],
)
deterministic_csv(
    EVALUATION_ROOT / "Stage6_Failure_Mode_Mechanism_Probe_v0.1.csv",
    mechanism_table,
)

figure_png = RESULT_ROOT / "Stage6_Frozen_External_Validation_ROC_And_Calibration_v0.1.png"
figure_pdf = RESULT_ROOT / "Stage6_Frozen_External_Validation_ROC_And_Calibration_v0.1.pdf"
if not figure_png.is_file() or not figure_pdf.is_file():
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for column, target in enumerate(["APTOS_2019", "IDRiD"]):
        table = evaluation_tables[target]
        y = table["moderate_or_worse_dr"].astype(int).to_numpy()
        p = table["frozen_axis_probability"].astype(float).to_numpy()
        fpr, tpr, _ = roc_curve(y, p)
        auc = metric_records[target]["metrics"]["roc_auc"]
        auc_ci = metric_records[target]["metric_ci_95"]["roc_auc"]
        axes[0, column].plot(fpr, tpr, color="#1864ab", linewidth=2)
        axes[0, column].plot([0, 1], [0, 1], "--", color="#868e96")
        axes[0, column].set(
            title=f"{target}: frozen raw ROC\nAUC {auc:.3f} [{auc_ci[0]:.3f}, {auc_ci[1]:.3f}]",
            xlabel="False-positive rate", ylabel="True-positive rate", xlim=(0, 1), ylim=(0, 1),
        )
        cal = calibration_tables[target]
        axes[1, column].plot([0, 1], [0, 1], "--", color="#868e96")
        axes[1, column].plot(
            cal["mean_probability"], cal["observed_fraction_positive"],
            marker="o", color="#d9480f", linewidth=2,
        )
        axes[1, column].set(
            title=f"{target}: equal-frequency calibration\nECE {metric_records[target]['metrics']['ece_10']:.3f}",
            xlabel="Mean frozen probability", ylabel="Observed positive fraction",
            xlim=(0, 1), ylim=(0, 1),
        )
    fig.suptitle("Stage 6: one-time frozen external validation", fontsize=15)
    fig.tight_layout()
    fig.savefig(figure_png, dpi=180, bbox_inches="tight")
    fig.savefig(
        figure_pdf, bbox_inches="tight",
        metadata={"CreationDate": None, "ModDate": None, "Title": "Stage 6 frozen external validation"},
    )
    plt.close(fig)
assert figure_png.is_file() and figure_pdf.is_file()

passes = stage6_summary["meets_external_feasibility_floor"].astype(bool)
if passes.all():
    overall_decision = "BOTH_EDGES_MEET_DISCRIMINATION_FLOOR_DESPITE_STAGE5B_ABSTENTION"
elif (~passes).all():
    overall_decision = "BOTH_EDGES_FAIL_PRECOMMITTED_EXTERNAL_DISCRIMINATION_FLOOR"
else:
    overall_decision = "MIXED_EXTERNAL_VALIDATION_ONE_EDGE_MEETS_AND_ONE_FAILS_FLOOR"

report_lines = [
    "# Stage 6 — One-Time Frozen External Validation", "",
    f"- Stage 5B freeze: `{EXPECTED_STAGE5B_FREEZE_HASH}`",
    f"- Stage 6 protocol seal: `{seal_payload['seal_sha256']}`",
    "- Target labels opened only after Stage 6 seal: `True`",
    "- Model refit, adaptation, sign reversal, or threshold tuning: `False`", "",
    "## Primary results", "",
]
for target in ["APTOS_2019", "IDRiD"]:
    record = metric_records[target]
    auc = record["metrics"]["roc_auc"]
    ci = record["metric_ci_95"]["roc_auc"]
    report_lines.extend([
        f"### DeepDRiD → {target}", "",
        f"- n = {record['n']}; positive = {record['positive']}.",
        f"- Raw-orientation ROC AUC = {auc:.6f} [{ci[0]:.6f}, {ci[1]:.6f}].",
        f"- Stage 5B warning evaluation: `{record['stage5b_warning_evaluation']}`.",
        f"- External feasibility floor met: `{record['meets_precommitted_external_feasibility_floor']}`.", "",
    ])
report_lines.extend([
    "## Interpretation boundary", "",
    "This is a two-edge prospective feasibility validation. It evaluates the two frozen warnings descriptively; it does not validate a general transfer-performance predictor or a relative ranking rule. Any adaptation or new method must begin only after this frozen Stage 6 record is complete.", "",
    f"Overall decision: `{overall_decision}`.", "",
])
report_text = "\n".join(report_lines)
report_path = RESULT_ROOT / "Stage6_One_Time_External_Validation_Report_v0.1.md"
deterministic_text(report_path, report_text)

output_candidates = sorted([
    path for root in [EVALUATION_ROOT, RESULT_ROOT]
    for path in root.iterdir()
    if (
        path.is_file()
        and path != RUNTIME_STATE_PATH
        and path != FINAL_DECISION_PATH
        and "Output_Integrity_Manifest" not in path.name
    )
], key=lambda path: str(path))
output_integrity = pd.DataFrame([{
    "relative_path": str(path.relative_to(STAGE6_ROOT)),
    "size_bytes": int(path.stat().st_size),
    "sha256": sha256_file(path),
} for path in output_candidates])
output_integrity_path = RESULT_ROOT / "Stage6_Output_Integrity_Manifest_v0.1.csv"
deterministic_csv(output_integrity_path, output_integrity)

final_payload = {
    "stage": "Stage6",
    "decision": overall_decision,
    "stage5b_freeze_sha256": EXPECTED_STAGE5B_FREEZE_HASH,
    "stage6_protocol_seal_sha256": seal_payload["seal_sha256"],
    "authorised_target_labels_opened": True,
    "unauthorised_quarantine_files_opened": False,
    "target_performance_observed": True,
    "model_refit": False,
    "model_adapted": False,
    "orientation_reversed": False,
    "threshold_tuned": False,
    "primary_results": {
        target: {
            "roc_auc": metric_records[target]["metrics"]["roc_auc"],
            "roc_auc_ci_95": metric_records[target]["metric_ci_95"]["roc_auc"],
            "meets_external_feasibility_floor": metric_records[target][
                "meets_precommitted_external_feasibility_floor"
            ],
            "stage5b_warning_evaluation": metric_records[target]["stage5b_warning_evaluation"],
        }
        for target in ["APTOS_2019", "IDRiD"]
    },
    "output_integrity_manifest_path": str(output_integrity_path),
    "output_integrity_manifest_sha256": sha256_file(output_integrity_path),
    "analysis_completed_utc": seal_payload["sealed_utc"],
    "next_step": (
        "LOCK_STAGE6_AND_REASSESS_PROJECT_FEASIBILITY_BEFORE_ANY_POST_UNSEAL_ADAPTATION"
    ),
}
final_payload["final_record_sha256"] = sha256_json(final_payload)
if FINAL_DECISION_PATH.is_file():
    with FINAL_DECISION_PATH.open("r", encoding="utf-8") as handle:
        existing_final = json.load(handle)
    existing_without_claim = dict(existing_final)
    existing_claim = existing_without_claim.pop("final_record_sha256")
    assert sha256_json(existing_without_claim) == existing_claim
    assert existing_final == final_payload
    final_payload = existing_final
else:
    atomic_json(FINAL_DECISION_PATH, final_payload)

runtime_state.update({
    "stage6_complete": True,
    "stage6_final_record": str(FINAL_DECISION_PATH),
    "stage6_final_record_sha256": final_payload["final_record_sha256"],
    "model_refit": False,
    "threshold_tuned": False,
    "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("\n================ STAGE 6 EXTERNAL VALIDATION COMPLETE ================")
display(stage6_summary)
display(mechanism_table)
print("\nDecision:", overall_decision)
print("Final Stage 6 record:", FINAL_DECISION_PATH)
print("Final record hash:", final_payload["final_record_sha256"])
print("Target sealed-label files accessed: True (authorised after protocol seal)")
print("Unauthorised quarantine files accessed: False")
print("Model refit/adaptation/sign reversal/threshold tuning: False")
print("\nSTOP. Reassess the complete Stage 5A–6 evidence before any adaptation work.")



================ STAGE 6 EXTERNAL VALIDATION COMPLETE ================


,edge_id,target,n,positive,prevalence,roc_auc,roc_auc_ci_95_lower,roc_auc_ci_95_upper,average_precision,balanced_accuracy_at_0_5,sensitivity_at_0_5,specificity_at_0_5,brier_score,ece_10,stage5b_abstain,meets_external_feasibility_floor,warning_evaluation
0,DDR_APTOS_v0.1,APTOS_2019,1080,432,0.400000,0.678434,0.644969,0.711861,0.640803,0.625772,0.634259,0.617284,0.323734,0.300644,True,False,OBSERVED_DISCRIMINATION_BELOW_PRECOMMITTED_FEA...
1,DDR_IDRiD_v0.1,IDRiD,102,63,0.617647,0.782662,0.692715,0.862434,0.865760,0.636752,0.888889,0.384615,0.261180,0.251944,True,True,OBSERVED_DISCRIMINATION_MEETS_FLOOR_DESPITE_CO...


,edge_id,target,stage5b_dominant_transform,stage5b_dominant_failure_mode,anchor_images,fixed_threshold_errors,median_abs_logit_change_errors_minus_correct,gap_ci_95_exploratory,top_quartile_sensitivity_error_rate,remaining_sensitivity_error_rate,interpretation_scope
0,DDR_APTOS_v0.1,APTOS_2019,CONTRAST_065,COLOUR_CONTRAST_MISMATCH,64,18,-0.636226,"[-2.146795971552207, 0.8418923966019135]",0.1875,0.312500,EXPLORATORY_MECHANISM_PROBE_NOT_PRIMARY_VALIDA...
1,DDR_IDRiD_v0.1,IDRiD,GAUSSIAN_NOISE_S004,BLUR_NOISE_SENSITIVITY_MISMATCH,64,16,-2.488388,"[-5.0709799450408575, 6.413347054990399]",0.3125,0.229167,EXPLORATORY_MECHANISM_PROBE_NOT_PRIMARY_VALIDA...



Decision: MIXED_EXTERNAL_VALIDATION_ONE_EDGE_MEETS_AND_ONE_FAILS_FLOOR
Final Stage 6 record: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Prospective_Retinal_Blind_Test_v0.1/Stage6_One_Time_Target_Unsealing_And_External_Validation_v0.1/02_Results/Stage6_External_Validation_Complete_v0.1.json
Final record hash: cbe17046eb97a191928d93dc0061a33ef52b630377b3af84103fa530568cb326
Target sealed-label files accessed: True (authorised after protocol seal)
Unauthorised quarantine files accessed: False
Model refit/adaptation/sign reversal/threshold tuning: False

STOP. Reassess the complete Stage 5A–6 evidence before any adaptation work.


## Stop boundary

After the final completion banner appears, the prospective two-edge experiment is locked. Do not modify these Stage 6 records. Any post-unseal method development, adaptation, or expanded multi-edge validation must live in a new, explicitly labelled post-unseal stage and must never be presented as part of the prospective blind evaluation.
